In [1]:
from utils.util import *
import geopandas as gpd
from shapely.geometry import shape
import folium
import os
import sys
import time
import pandas as pd
fileType = 'band'
years = ['2010', '2011', '2012', '2013']
includeMetadata = True

# For albedo use band calculation https://chatgpt.com/share/67631932-ca54-8003-aff0-515d7ca98ed6
# For DEM use one terrain global SRTM 1 Arc-Second Global
# For land cover...

C:\Users\jesse\miniconda3\envs\ml\lib\site-packages\pyproj\__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


Directory './Unprocessed' already exists.
Directory './Data' already exists.
Directory './RawClippedRasters' already exists.
Directory './Data/LST' already exists.
Directory './Data/NDVI' already exists.
Directory './Data/NDWI' already exists.
Directory './Data/Land_Cover' already exists.
Directory './Data/Albedo' already exists.
Directory './Data/DEM' already exists.
Directory './RawClippedRasters/LST' already exists.
Directory './RawClippedRasters/NDVI' already exists.
Directory './RawClippedRasters/NDWI' already exists.
Directory './RawClippedRasters/Land_Cover' already exists.
Directory './RawClippedRasters/Albedo' already exists.
Directory './RawClippedRasters/DEM' already exists.
Logging in...


Login Successful, API Key Received!


In [2]:
# Cell 2: Load shape file
shapefile_folder = "./Data/area_shp/"
shapefile = "Polygon_San_Antonio_TX.shp"
city = shapefile.replace('Polygon_', '').replace('.shp', '')
aoi_geodf = gpd.read_file(shapefile_folder + shapefile)
aoi_geodf = aoi_geodf.to_crs("EPSG:4326")
if aoi_geodf.empty:
    sys.exit("Error: Shapefile contains no data.")
print("Shapefile loaded successfully.")

CRSError: Invalid projection: EPSG:4326: (Internal Proj Error: proj_create: no database context specified)

In [3]:
centroid = aoi_geodf.geometry.centroid.iloc[0]
m = folium.Map(
    location=[centroid.y, centroid.x], 
    zoom_start=9, tiles="openstreetmap", width="100%", height="100%", attributionControl=0
)
# Cell 4: Add Polygon to Map
folium.GeoJson(aoi_geodf).add_to(m)
m

C:\Users\jesse\AppData\Local\Temp\ipykernel_17840\3196744355.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = aoi_geodf.geometry.centroid.iloc[0]


In [2]:
# Define search payload (if filtering is needed, adjust accordingly)
dataset_search_payload = {}

# Send request to dataset search endpoint
datasets = sendRequest(serviceUrl + "dataset-search", dataset_search_payload, apiKey)

# Print the names of available datasets
for ds in datasets:
    # print(ds.keys())
    print(f"Dataset Name: {ds['datasetCategoryName']} + -> {ds['datasetAlias']}, {ds['sceneCount']}")

Dataset Name: 2004 Events + -> event1058, 0
Dataset Name: 2005 Events + -> event741, 1270103
Dataset Name: 2007 Events + -> event19, 0
Dataset Name: 2007 Events + -> event20, 0
Dataset Name: 2007 Events + -> event28, 1270103
Dataset Name: 2008 Events + -> event148, 0
Dataset Name: 2008 Events + -> event137, 0
Dataset Name: 2008 Events + -> event161, 0
Dataset Name: 2008 Events + -> event222, 0
Dataset Name: 2008 Events + -> event207, 1270103
Dataset Name: 2008 Events + -> event245, 0
Dataset Name: 2008 Events + -> event273, 0
Dataset Name: 2008 Events + -> event303, 1270103
Dataset Name: 2008 Events + -> event272, 0
Dataset Name: 2009 Events + -> event326, 1270103
Dataset Name: 2009 Events + -> event314, 0
Dataset Name: 2009 Events + -> event328, 0
Dataset Name: 2009 Events + -> event325, 0
Dataset Name: 2009 Events + -> event323, 1270103
Dataset Name: 2009 Events + -> event330, 0
Dataset Name: 2009 Events + -> event331, 1270103
Dataset Name: 2009 Events + -> event336, 1270103
Dataset 

In [3]:
datasetName = 'nlcd_collection_lndcov'
# Cell 5: Define Scene Search Parameters, temporal is inclusive
spatialFilter = {
    'filterType': 'mbr',
    'lowerLeft': {
        'latitude': aoi_geodf.geometry.bounds.miny[0],
        'longitude': aoi_geodf.geometry.bounds.minx[0]
    },
    'upperRight': {
        'latitude': aoi_geodf.geometry.bounds.maxy[0],
        'longitude': aoi_geodf.geometry.bounds.maxx[0]
    }
}

temporalFilter = {'start': '2018-10-01', 'end': '2020-12-31'}
search_payload = {
    'datasetName': datasetName,
}

NameError: name 'aoi_geodf' is not defined

In [37]:
# Cell 6: Search for Scenes
scenes = sendRequest(serviceUrl + "scene-search", search_payload, apiKey)
print(scenes)
# pd.json_normalize(scenes['results'])

{'results': [], 'recordsReturned': 0, 'totalHits': 0, 'totalHitsAccuracy': 'approximate', 'isCustomized': False, 'numExcluded': 0, 'startingNumber': 0, 'nextRecord': 0}


In [6]:
# Cell 7: Collect Entity IDs
entityIds = [result['entityId'] for result in scenes['results'] if result['options']['bulk']]

In [7]:
# Cell 8: Prepare Scene List for Download
listId = f"temp_{datasetName}_list"
scn_list_add_payload = {
    "listId": listId,
    'idField': 'entityId',
    "entityIds": entityIds,
    "datasetName": datasetName
}
sendRequest(serviceUrl + "scene-list-add", scn_list_add_payload, apiKey)

2

In [8]:
# Cell 9: Prepare Download Options
download_opt_payload = {
    "listId": listId,
    "datasetName": datasetName
}
products = sendRequest(serviceUrl + "download-options", download_opt_payload, apiKey)
pd.json_normalize(products)


,id,downloadName,displayId,entityId,datasetId,available,filesize,productName,productCode,bulkAvailable,downloadSystem,secondaryDownloads,fileGroups
0,5e83d14fec7cae84,None,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,True,1008269944,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
1,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
2,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
3,632210d4770592cf,None,LC08_L2SP_027039_20200412_20200822_02_T1,LC80270392020103LGN00,5e83d14f2fc39685,False,1008269944,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
4,5e83d14fec7cae84,None,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,True,983572232,Landsat Collection 2 Level-2 Product Bundle,D694,True,ls_zip,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
5,6448198cc7b442a4,C2L2 Tile Product Files,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D693,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
6,6448198c62023764,C2L2 Tile Product Files,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,True,0,Landsat Collection 2 Level-2 Band File,D691,True,folder,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None
7,632210d4770592cf,None,LC08_L2SP_027040_20200412_20200822_02_T1,LC80270402020103LGN00,5e83d14f2fc39685,False,983572232,Landsat Collection 2 Level-2 Product Bundle,D806,False,dds_ms,"[{'id': '5f85f041a2ea6695', 'downloadName': No...",None


In [9]:
# Cell 10: Collect Files to Download
downloads = []
for product in products:
    if product["secondaryDownloads"]:
        for secDownload in product["secondaryDownloads"]:
            if secDownload["bulkAvailable"] and any(band in secDownload['displayId'] for band in bandNames):
                downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})
            if includeMetadata and secDownload['displayId'].endswith('_MTL.txt'):
                downloads.append({"entityId": secDownload["entityId"], "productId": secDownload["id"]})


In [10]:
# Cell 11: Submit Download Request
download_req_payload = {
    "downloads": downloads,
    "label": listId
}
download_request_results = sendRequest(serviceUrl + "download-request", download_req_payload, apiKey)

In [11]:
# Cell 12: Download Files
for result in download_request_results['availableDownloads']:
    runDownload(threads, result['url'])
for i, t in enumerate(threads):
    t.join()

    Downloading: LC08_L2SP_027039_20200412_20200822_02_T1_MTL.txt...
    Downloading: LC08_L2SP_027039_20200412_20200822_02_T1_SR_B3.TIF...
    Downloading: LC08_L2SP_027039_20200412_20200822_02_T1_SR_B4.TIF...
    Downloading: LC08_L2SP_027039_20200412_20200822_02_T1_ST_B10.TIF...
    Downloading: LC08_L2SP_027039_20200412_20200822_02_T1_SR_B5.TIF...
    Downloading: LC08_L2SP_027040_20200412_20200822_02_T1_MTL.txt...
    Downloading: LC08_L2SP_027040_20200412_20200822_02_T1_SR_B3.TIF...
    Downloading: LC08_L2SP_027040_20200412_20200822_02_T1_SR_B4.TIF...
    Downloading: LC08_L2SP_027040_20200412_20200822_02_T1_SR_B5.TIF...
    Downloading: LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF...


In [12]:
# Cell 13: Clean Up
remove_scnlst_payload = {"listId": listId}
sendRequest(serviceUrl + "scene-list-remove", remove_scnlst_payload, apiKey)

In [13]:
# Cell 15: Verify Downloads
import rasterio
from shapely.geometry import box

usableTIFFs = []
print(os.listdir(unprocessed_dir))
# Ensure the shapefile matches the TIF CRS
for tif_file in os.listdir(unprocessed_dir):
    if tif_file.endswith(".TIF"):
        tif_path = unprocessed_dir + '/' + tif_file
        with rasterio.open(tif_path) as src:
            # Reproject shapefile to match TIF file CRS
            aoi_geodf_proj = aoi_geodf.to_crs(src.crs)
            tif_bounds = box(*src.bounds)
            
            # Check containment
            for idx, geom in enumerate(aoi_geodf_proj.geometry):
                if tif_bounds.contains(geom):
                    usableTIFFs.append(tif_file)
                    print(f"Polygon {city} is fully inside {tif_file}")
                else:
                    print(f"Polygon {city} is NOT fully inside {tif_file}")
print(usableTIFFs)

['LC08_L2SP_027039_20200412_20200822_02_T1_MTL.txt', 'LC08_L2SP_027039_20200412_20200822_02_T1_SR_B3.TIF', 'LC08_L2SP_027039_20200412_20200822_02_T1_SR_B4.TIF', 'LC08_L2SP_027039_20200412_20200822_02_T1_SR_B5.TIF', 'LC08_L2SP_027039_20200412_20200822_02_T1_ST_B10.TIF', 'LC08_L2SP_027040_20200412_20200822_02_T1_MTL.txt', 'LC08_L2SP_027040_20200412_20200822_02_T1_SR_B3.TIF', 'LC08_L2SP_027040_20200412_20200822_02_T1_SR_B4.TIF', 'LC08_L2SP_027040_20200412_20200822_02_T1_SR_B5.TIF', 'LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF']
Polygon San_Antonio_TX is NOT fully inside LC08_L2SP_027039_20200412_20200822_02_T1_SR_B3.TIF
Polygon San_Antonio_TX is NOT fully inside LC08_L2SP_027039_20200412_20200822_02_T1_SR_B4.TIF
Polygon San_Antonio_TX is NOT fully inside LC08_L2SP_027039_20200412_20200822_02_T1_SR_B5.TIF
Polygon San_Antonio_TX is NOT fully inside LC08_L2SP_027039_20200412_20200822_02_T1_ST_B10.TIF
Polygon San_Antonio_TX is fully inside LC08_L2SP_027040_20200412_20200822_02_T1_SR_B

In [14]:
goodCoordinates = clipUnprocessedRasters(usableTIFFs, aoi_geodf_proj)

Masked and reprojected TIF saved as ./Unprocessed/Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_SR_B3.TIF
Masked and reprojected TIF saved as ./Unprocessed/Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_SR_B4.TIF
Masked and reprojected TIF saved as ./Unprocessed/Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_SR_B5.TIF
Masked and reprojected TIF saved as ./Unprocessed/Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF


In [15]:
for file in os.listdir(unprocessed_dir):
    if ".txt" in file or "Clipped_" in file:
        date, band, coordinate = getMetaFromLandsatTIRs(file)
        if coordinate not in goodCoordinates:
            continue
        print(date, band)
        if band == 'B10':
            moveToRaw(file, 'LST', date, city)
        if band == 'B3':
            moveToRaw(file, 'NDWI', date, city)
        if band == 'B4':
            moveToRaw(file, 'NDVI', date, city)
        if band == 'B5':
            moveToRaw(file, 'NDVI', date, city)
            moveToRaw(file, 'NDWI', date, city)
        if band == 'MTL':
            moveToRaw(file, 'LST', date, city)
            moveToRaw(file, 'NDVI', date, city)
            moveToRaw(file, 'NDWI', date, city)            

2020-04-12 B3
2020-04-12 B4
2020-04-12 B5
2020-04-12 B10
2020-04-12 MTL


In [16]:
clear_folder(unprocessed_dir)

Deleted file: ./Unprocessed\Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_SR_B3.TIF
Deleted file: ./Unprocessed\Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_SR_B4.TIF
Deleted file: ./Unprocessed\Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_SR_B5.TIF
Deleted file: ./Unprocessed\Clipped_LC08_L2SP_027040_20200412_20200822_02_T1_ST_B10.TIF
Deleted file: ./Unprocessed\LC08_L2SP_027039_20200412_20200822_02_T1_MTL.txt
Deleted file: ./Unprocessed\LC08_L2SP_027039_20200412_20200822_02_T1_SR_B3.TIF
Deleted file: ./Unprocessed\LC08_L2SP_027039_20200412_20200822_02_T1_SR_B4.TIF
Deleted file: ./Unprocessed\LC08_L2SP_027039_20200412_20200822_02_T1_SR_B5.TIF
Deleted file: ./Unprocessed\LC08_L2SP_027039_20200412_20200822_02_T1_ST_B10.TIF
Deleted file: ./Unprocessed\LC08_L2SP_027040_20200412_20200822_02_T1_MTL.txt
Deleted file: ./Unprocessed\LC08_L2SP_027040_20200412_20200822_02_T1_SR_B3.TIF
Deleted file: ./Unprocessed\LC08_L2SP_027040_20200412_20200822_02_T1_SR_B4.TIF
Deleted file: ./Unproc